<a href="https://colab.research.google.com/github/bsheese/387/blob/main/20_how_networks_learn/20_1_Network_Anatomy/20_1_1_Anatomy_of_a_Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# How Networks Learn: Part 4
## Building a Network

---

## What This Notebook Is About

You now have all the pieces of a neural network, built separately:

- **19_1_3**: a neuron is a weighted sum of its inputs plus a bias, passed through an activation function, and a layer of neurons is a matrix multiplication.
- **19_1_4**: one neuron can't solve XOR, but a hidden layer of two neurons can. We chose the hidden weights by hand.
- **20_0**: autograd gives the slope of the loss for every weight, and gradient descent uses the slopes to improve the weights.

This notebook puts them together with PyTorch's building blocks, and lets gradient descent find the hidden layer's weights on its own.

**What you will learn:**
1. `nn.Linear`, PyTorch's layer of neurons, and how it matches the weighted sums from 19_1_3
2. `nn.Module`, the pattern every PyTorch model follows
3. Why the hidden layer uses the ReLU activation instead of 19_1_4's step function
4. How a network with a hidden layer learns XOR, and what its hidden layer computes
5. That training doesn't always succeed, and how much the starting weights matter
6. How to look inside any model, including ResNet-18 from 19_0

**Optional reference.** If you took CS 377, you have seen these ideas before in [18_2_3](https://github.com/bsheese/cs377/blob/main/18_classification/18_2_LogisticRegression/18_2_3_LogReg_to_NeuralNetworks.ipynb). If you haven't, those notebooks cover the same ideas on other datasets, in more depth. You don't need them for this course.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

X = torch.tensor([[0., 0.],
                  [0., 1.],
                  [1., 0.],
                  [1., 1.]])
y = torch.tensor([[0.], [1.], [1.], [0.]])    # XOR: 1 when exactly one input is 1
print(X.shape, y.shape)

These are the same four XOR points as in 19_1_4. `y` has shape (4, 1), one column, because the network will give one output per point.

---

## Section 1: `nn.Linear`, a Layer of Neurons

`nn.Linear(n_inputs, n_outputs)` is a layer of `n_outputs` neurons, each with one weight per input and a bias. It creates the weights for you (starting at small random values) and computes the weighted sums. You've already used one: the last layer of the Pets model in 19_0 was `nn.Linear(512, 4)`.

In [ ]:
torch.manual_seed(0)
layer = nn.Linear(2, 3)            # 3 neurons, each with 2 weights and a bias
print('weight:', layer.weight.shape)
print('bias:  ', layer.bias.shape)

The weight matrix has one row per neuron, shape (3, 2), the same layout as the Pets model's (4, 512) matrix in 19_1_3. Calling the layer on the four points computes `X @ weight.T + bias`, the matrix multiplication from 19_1_3:

In [ ]:
out = layer(X)
by_hand = X @ layer.weight.T + layer.bias
print(out.shape)
print('same as by hand:', torch.allclose(out, by_hand))

The output has shape (4, 3): one row per point and one column per neuron. Its weights were created with `requires_grad=True`, so autograd tracks everything computed from them.

---

## Section 2: `nn.Module`, the Shape of Every Model

A PyTorch model is a Python class built on `nn.Module`, and it has two parts:

- **`__init__`** creates the layers and stores them as attributes. PyTorch finds every layer stored this way and collects its weights, so `model.parameters()` can hand them all to an optimizer.
- **`forward(x)`** says how an input becomes an output. You don't call it directly; calling `model(x)` runs it.

Here's the single neuron from 19_1_3 (logistic regression) as a module: one `nn.Linear` with one output, followed by sigmoid.

In [ ]:
class OneNeuron(nn.Module):
    def __init__(self):
        super().__init__()                 # always the first line: sets up the nn.Module machinery
        self.linear = nn.Linear(2, 1)

    def forward(self, x):
        return torch.sigmoid(self.linear(x))

torch.manual_seed(0)
neuron = OneNeuron()
print(neuron)
print('probabilities before training:', neuron(X).detach().squeeze().numpy().round(3))

### Training it on XOR

The loss for a yes/no problem is **binary cross-entropy**, `nn.BCELoss`. It's the cross-entropy loss from 19_1_4 for two classes: for each point, $-\log$ of the probability the model gave to the right answer, averaged over the points. The training loop is the one from 20_0_3, with `torch.optim.SGD` taking the steps.

In [ ]:
loss_fn = nn.BCELoss()
optimizer = torch.optim.SGD(neuron.parameters(), lr=0.5)

for step in range(2000):
    optimizer.zero_grad()
    loss = loss_fn(neuron(X), y)
    loss.backward()
    optimizer.step()

print(f'final loss: {loss.item():.4f}   (-log(0.5) = {-np.log(0.5):.4f})')
print('probabilities after training:', neuron(X).detach().squeeze().numpy().round(3))

The same result as scikit-learn's logistic regression in 19_1_4: after 2,000 steps the neuron gives every point a probability of 0.5, and the loss is $-\log(0.5)$. A single neuron draws one straight line, no straight line separates XOR, and "50/50 everywhere" is the best it can do by the loss.

---

## Section 3: The Hidden Layer Needs an Activation It Can Learn With

In 19_1_4 the hidden neurons used a **step** activation: 1 if the weighted sum is above 0, otherwise 0. It made the arithmetic easy, but gradient descent can't learn with it. Use the nudge test from 20_0_2 on a step function:

In [ ]:
def step(z):
    return (z > 0).float()

def nudge_slope(f, z, h=0.001):
    return (f(z + h) - f(z - h)) / (2 * h)

for z in [-2.0, -0.5, 0.5, 2.0]:
    print(f'slope of step at z = {z:4}: {nudge_slope(step, torch.tensor(z)).item()}')

The slope is 0 everywhere (except exactly at 0, where the step jumps). Because rates multiply (20_0_2), a slope of 0 anywhere in the chain makes the slope of the loss 0 for every weight before it. With step activations, every hidden weight would have a slope of 0, and gradient descent would never move them.

The activation used in most networks today is **ReLU** (rectified linear unit): $\text{ReLU}(z) = \max(0, z)$. It passes positive values through unchanged and turns negative values into 0.

In [ ]:
zs = torch.linspace(-3, 3, 200)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
ax1.plot(zs, step(zs))
ax1.set_title('step: slope 0 everywhere')
ax2.plot(zs, torch.relu(zs))
ax2.set_title('ReLU: slope 1 for positive z')
for ax in (ax1, ax2):
    ax.set_xlabel('weighted sum z')
plt.tight_layout()
plt.show()

for z in [-2.0, -0.5, 0.5, 2.0]:
    print(f'slope of ReLU at z = {z:4}: {nudge_slope(torch.relu, torch.tensor(z)).item():.2f}')

ReLU's slope is 1 wherever the weighted sum is positive, so slopes can pass back through it to the weights before it. It's still bent, not a straight line, and that bend is what lets a hidden layer compute new features that a straight line couldn't. (If the hidden layer had no activation at all, two layers of weighted sums would just add up to one weighted sum, and the network would be a single straight line again.)

---

## Section 4: A Network That Learns XOR

The network below has a hidden layer of 4 ReLU neurons and an output layer of one sigmoid neuron:

In [ ]:
class XORNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(2, 4)    # 2 inputs -> 4 hidden neurons
        self.output = nn.Linear(4, 1)    # 4 hidden values -> 1 output

    def forward(self, x):
        h = torch.relu(self.hidden(x))           # the hidden layer's 4 new features
        return torch.sigmoid(self.output(h))     # one neuron on top, as in 19_1_3

for name, p in XORNet().named_parameters():
    print(f'{name:14} shape {str(list(p.shape)):8} {p.numel():3} numbers')
print('total:', sum(p.numel() for p in XORNet().parameters()))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.axis('off')
ax.set_aspect('equal')
positions = {'input': (0.5, [2.0, 1.0]), 'hidden': (2.5, [3.0, 2.0, 1.0, 0.0]), 'output': (4.5, [1.5])}
names = {'input': ['x1', 'x2'], 'hidden': ['h1', 'h2', 'h3', 'h4'], 'output': ['p']}
for (a, b) in [('input', 'hidden'), ('hidden', 'output')]:
    for ya in positions[a][1]:
        for yb in positions[b][1]:
            ax.plot([positions[a][0], positions[b][0]], [ya, yb], color='#BBB', lw=0.8, zorder=1)
for layer_name, (x_pos, ys) in positions.items():
    for y_pos, label in zip(ys, names[layer_name]):
        ax.add_patch(plt.Circle((x_pos, y_pos), 0.22, color='#D6EAF8', ec='#333', zorder=2))
        ax.text(x_pos, y_pos, label, ha='center', va='center', zorder=3)
ax.text(0.5, 3.7, 'inputs', ha='center', fontweight='bold')
ax.text(2.5, 3.7, 'hidden layer (ReLU)', ha='center', fontweight='bold')
ax.text(4.5, 3.7, 'output (sigmoid)', ha='center', fontweight='bold')
ax.text(1.5, -0.9, 'nn.Linear(2, 4)\n8 weights + 4 biases', ha='center', fontsize=9)
ax.text(3.5, -0.9, 'nn.Linear(4, 1)\n4 weights + 1 bias', ha='center', fontsize=9)
ax.set_xlim(-0.2, 5.2)
ax.set_ylim(-1.3, 4.0)
plt.show()

Every line in the diagram is one weight. The hidden layer's weight matrix has shape (4, 2), one row per hidden neuron, and the output layer's has shape (1, 4). There are 17 numbers in all for gradient descent to set. Training uses exactly the same loop as the single neuron:

In [ ]:
def train(model, steps=2000, lr=0.5):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    losses = []
    for step in range(steps):
        optimizer.zero_grad()
        loss = loss_fn(model(X), y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses

torch.manual_seed(42)
net = XORNet()
losses = train(net)

plt.figure(figsize=(7, 3.5))
plt.plot(losses)
plt.xlabel('step')
plt.ylabel('loss (binary cross-entropy)')
plt.show()

with torch.no_grad():
    p = net(X)
for xi, yi, pi in zip(X, y, p):
    print(f'x = {xi.tolist()}   target {int(yi.item())}   probability {pi.item():.3f}')

The loss falls to nearly 0, and the network gives each point a probability very close to its target. Nobody chose the hidden weights; gradient descent found them, starting from random values.

Here is the network's answer for every point in the square around the four XOR points. The shading shows where the network says 1 (red) or 0 (blue), and the dashed curve is where its probability is exactly 0.5.

In [ ]:
grid_x, grid_y = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
grid = torch.tensor(np.column_stack([grid_x.ravel(), grid_y.ravel()]), dtype=torch.float32)
with torch.no_grad():
    probs = net(grid).numpy().reshape(grid_x.shape)

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.contourf(grid_x, grid_y, probs, levels=[0, 0.5, 1], colors=['#D6EAF8', '#FADBD8'])
ax.contour(grid_x, grid_y, probs, levels=[0.5], colors='k', linestyles='--')
for xi, yi in zip(X, y):
    ax.scatter(xi[0], xi[1], s=250, c='#C0392B' if yi.item() == 1 else '#2980B9', edgecolors='black', zorder=3)
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.set_title("XORNet's decision boundary")
plt.show()

The boundary isn't a single straight line. It's made of two lines that form a band: the two blue corners fall inside the band and the two red corners fall outside it. No single neuron can draw a boundary like that, because each neuron contributes one straight line, and here the hidden neurons' lines have been combined.

### What the hidden layer computes

As in 19_1_4, the hidden layer turns each point's two inputs into new features, here four of them. These are the values the output neuron actually sees:

In [ ]:
with torch.no_grad():
    hidden_values = torch.relu(net.hidden(X))
print('  x1 x2  |  h1     h2     h3     h4    | target')
for xi, hi, yi in zip(X, hidden_values, y):
    print(f'  {int(xi[0])}  {int(xi[1])}   | ' + '  '.join(f'{v:5.2f}' for v in hi) + f'  |   {int(yi.item())}')
print('output weights:', net.output.weight.detach().numpy().round(2), ' bias:', net.output.bias.detach().numpy().round(2))

The learned features are harder to read than 19_1_4's "at least one" and "both", and they'll differ each time the network is trained from a different start. They do the same job, though. The output neuron is still one weighted sum followed by sigmoid, and with these four features its weighted sum is high for the two target-1 points and low for the other two. The hidden layer has rearranged the points so that a single neuron's straight boundary works, and it learned to do that from the loss alone.

---

## Section 5: Checking a Hidden Weight's Slope

The slope for a hidden weight passes through more steps than any slope in 20_0: the weight changes a hidden neuron's weighted sum, which goes through ReLU, which feeds the output neuron's weighted sum, which goes through sigmoid, which feeds the loss. Autograd multiplies the rates of all those steps. We can check its answer with the nudge test. The next cell takes a fresh, untrained network, finds the hidden weight with the largest slope, and nudges it:

In [ ]:
torch.manual_seed(1)
fresh = XORNet()

loss = loss_fn(fresh(X), y)
fresh.zero_grad()
loss.backward()
grads = fresh.hidden.weight.grad
row, col = divmod(grads.abs().argmax().item(), grads.shape[1])   # position of the largest slope
from_autograd = grads[row, col].item()

w = fresh.hidden.weight
with torch.no_grad():
    original = w[row, col].item()
    w[row, col] = original + 0.001
    up = loss_fn(fresh(X), y).item()
    w[row, col] = original - 0.001
    down = loss_fn(fresh(X), y).item()
    w[row, col] = original
print(f'hidden weight [{row}, {col}]:  nudge test {(up - down) / 0.002:.4f}   autograd {from_autograd:.4f}')

They agree. This is the check from the end of 20_0_2, done on a network with a hidden layer: backpropagation through five steps gives the same slope as nudging the weight directly. In a network with millions of weights it does that for every weight in one backward pass.

---

## Section 6: Training Doesn't Always Work

The network above was trained from one particular set of starting weights, chosen by `torch.manual_seed(42)`. The next cell trains 20 copies of XORNet, each from different random starting weights, with everything else the same, and checks how many get all four points right.

In [ ]:
results = []
for seed in range(20):
    torch.manual_seed(seed)
    model = XORNet()
    train(model)
    with torch.no_grad():
        right = ((model(X) > 0.5).float() == y).sum().item()
    results.append(right)

print('points right, for each starting point:', results)
print(results.count(4), 'of 20 runs solved XOR')

Most runs solve XOR, but some get stuck with only two of the four points right, even though they have the same network, the same data and the same number of steps. Gradient descent only ever goes downhill from where it starts, so a bad starting point can lead it somewhere it can't improve on by small steps. With such a tiny network there's a specific way this happens with ReLU: if a hidden neuron's weighted sum is negative for all four points, its output is always 0 and its slope is always 0, so its weights never change. The neuron is "dead".

Larger networks, with many more neurons than the problem strictly needs, get stuck much less often, which is one reason networks in practice are much bigger than the smallest one that could work. Unit 21 comes back to how starting weights and the choice of optimizer affect training.

---

## Section 7: `nn.Sequential`, and Looking Inside Any Model

When a model just runs its layers one after another, `nn.Sequential` saves writing a class. This is the same network as `XORNet`:

In [ ]:
seq = nn.Sequential(
    nn.Linear(2, 4),
    nn.ReLU(),
    nn.Linear(4, 1),
    nn.Sigmoid(),
)
print(seq)

(`nn.ReLU()` and `nn.Sigmoid()` are the layer versions of `torch.relu` and `torch.sigmoid`, so they can go in the list.) Use `nn.Sequential` for straight stacks like this, and write an `nn.Module` class when the forward pass needs anything more, such as using an input twice or combining two branches.

Every model can list its weights by name with `named_parameters()`, and `state_dict()` returns all of them as a dictionary of tensors. A state dictionary is what `torch.save` writes to a file. The `pets4_head.pt` file from 19_0 was the state dictionary of one `nn.Linear(512, 4)`, which is why it held a `weight` and a `bias`.

In [ ]:
for name, tensor in seq.state_dict().items():
    print(f'{name:10} {list(tensor.shape)}')

The same tools work on a real network. Here is ResNet-18, the model from 19_0. Creating it without downloading the trained weights is instant, and the structure is the same.

In [ ]:
from torchvision import models
resnet = models.resnet18()          # the same architecture, with random weights

n_weights = sum(p.numel() for p in resnet.parameters())
print(f'ResNet-18 has {n_weights:,} numbers in {len(list(resnet.parameters()))} weight tensors')
print('its last layer:', resnet.fc)
print()
print('the first five weight tensors:')
for name, p in list(resnet.named_parameters())[:5]:
    print(f'  {name:22} {list(p.shape)}')

The last layer, `fc`, is an `nn.Linear(512, 1000)`: one neuron for each ImageNet category. In 19_0 we replaced it with `nn.Linear(512, 4)`. The first weight tensor, `conv1.weight`, has four dimensions, (64, 3, 7, 7). It belongs to a **convolutional** layer, which looks at small 7 × 7 patches of the 3 color channels at a time; unit 22 explains how those layers work. Everything in between is more layers, and all 11.7 million numbers are trained the same way as XORNet's 17: autograd finds their slopes, and the optimizer steps them downhill.

---

## Summary

- `nn.Linear(n_inputs, n_outputs)` is a layer of neurons: a weight matrix of shape (n_outputs, n_inputs) and one bias per neuron. It computes `x @ weight.T + bias`, the matrix multiplication from 19_1_3.
- A model is an `nn.Module`: `__init__` creates the layers and `forward` says how they're used. `model.parameters()` gives every weight to the optimizer.
- A single neuron trained on XOR ends at a probability of 0.5 everywhere, as in 19_1_4.
- The step activation has a slope of 0, so gradient descent can't train weights before it. **ReLU**, $\max(0, z)$, has a slope of 1 for positive inputs and is still bent, which is what a hidden layer needs.
- XORNet (2 inputs, 4 hidden ReLU neurons, 1 sigmoid output, 17 numbers) learns XOR with **binary cross-entropy** and gradient descent. Its hidden layer learns features that make the points separable for the output neuron.
- The nudge test confirms autograd's slope for a hidden weight, through five steps of the network.
- Training can get stuck depending on the starting weights: some of 20 runs failed. Bigger networks get stuck less often.
- `nn.Sequential` stacks layers in order; `named_parameters()` and `state_dict()` list a model's weights. ResNet-18 is the same idea at scale: 11.7 million numbers, ending in an `nn.Linear`.

**Next:** unit 21 trains networks on real data: the full training loop with batches, validation and overfitting, starting with the Wisconsin breast cancer data in 21_0_1.